<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/H5_OmniFusion_Colab_Runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 H5-OmniFusion Pipeline - RESUME FROM CHECKPOINT

**Each participant is saved as a SEPARATE H5 file!**

Output: `355.h5`, `356.h5`, `t_28.h5`, etc.

---

## Quick Start
1. Run all cells (no config needed!)
2. Pipeline auto-detects existing files and resumes
3. Each participant = one H5 file in Google Drive

In [1]:
# 1. Clone fresh
!git clone https://github.com/nithin12342/phase2.git /content/phase2

# 2. Check Commit (MUST BE db865a3)
!cd /content/phase2 && git rev-parse --short HEAD

# 3. IF commit is correct -> Run All Cells

Cloning into '/content/phase2'...
remote: Enumerating objects: 833, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 833 (delta 147), reused 192 (delta 86), pack-reused 553 (from 1)
Receiving objects: 100% (833/833), 12.16 MiB | 15.68 MiB/s, done.
Resolving deltas: 100% (370/370), done.
05b6dd8


In [2]:
!git pull origin main

fatal: not a git repository (or any of the parent directories): .git


In [3]:
import os
# 1. Clone if it doesn't exist, or Pull if it does
if not os.path.exists("phase2"):
    !git clone https://github.com/nithin12342/phase2.git
    %cd phase2
else:
    %cd phase2
    !git pull origin main

# 2. Verify we are in the right place (should show .git)
!ls -la

/content/phase2
From https://github.com/nithin12342/phase2
 * branch            main       -> FETCH_HEAD
Already up to date.
total 136
drwxr-xr-x 8 root root  4096 Jan 22 05:07  .
drwxr-xr-x 1 root root  4096 Jan 22 05:07  ..
-rw-r--r-- 1 root root 13099 Jan 22 05:07  108_step_audit_matrix.md
-rw-r--r-- 1 root root  7468 Jan 22 05:07  analyze_h5.py
-rw-r--r-- 1 root root   855 Jan 22 05:07  analyze.py
-rw-r--r-- 1 root root  3226 Jan 22 05:07  check_h5.py
-rw-r--r-- 1 root root   658 Jan 22 05:07  docker-compose.yml
drwxr-xr-x 5 root root  4096 Jan 22 05:07  docs
-rw-r--r-- 1 root root  8180 Jan 22 05:07  Download_ViT_DINO_to_Drive.ipynb
-rw-r--r-- 1 root root  1682 Jan 22 05:07  fix_nltk.py
-rw-r--r-- 1 root root 12306 Jan 22 05:07  generate_audit_report.py
-rw-r--r-- 1 root root   999 Jan 22 05:07  gen_report.py
drwxr-xr-x 8 root root  4096 Jan 22 05:07  .git
-rw-r--r-- 1 root root   506 Jan 22 05:07  .gitignore
-rw-r--r-- 1 root root  3036 Jan 22 05:07  h5_audit_output.txt
-rw-r--r-

## Cell 1: Install Dependencies

In [4]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

!pip install -q transformers>=4.36.0 timm einops huggingface_hub --quiet
!pip install -q librosa soundfile praat-parselmouth noisereduce --quiet
!pip install -q mediapipe nltk vaderSentiment snownlp h5py tqdm --quiet
!pip install -q opensmile --quiet 2>/dev/null || echo "OpenSMILE fallback"
!pip install -q pandas tabulate --quiet

import nltk
try:
    nltk.download('vader_lexicon')
    nltk.download('punkt')
except:
    pass


print("\n✅ Dependencies installed. RESTART RUNTIME if needed.")

PyTorch: 2.9.0+cu126, CUDA: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 104.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 16.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.3/157.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.



✅ Dependencies installed. RESTART RUNTIME if needed.


## Cell 2: Mount Google Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3: Clone Repository

In [6]:
import os
REPO_URL = 'https://github.com/nithin12342/phase2.git'
REPO_DIR = '/content/phase2'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
print(f'Repository ready: {REPO_DIR}')

Already up to date.
Repository ready: /content/phase2


## Cell 4: Add Paths

In [7]:
import sys
PREPROCESSING_PATH = '/content/phase2/ml_pipeline/h5_omnifusion/preprocessing_and_feature_extraction'
SRC_PATH = '/content/phase2/ml_pipeline/h5_omnifusion/src'
for p in [PREPROCESSING_PATH, SRC_PATH]:
    if p not in sys.path: sys.path.insert(0, p)
print('Paths added')

Paths added


## Cell 5: Configuration

In [8]:
from dataclasses import dataclass
from typing import Tuple
import os, torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass
class Config:
    BASE_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
    DAIC_WOZ_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/DAIC-WOZ'
    EXTENDED_DAIC_WOZ_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/Extended-DAIC-WOZ/data'
    EATD_CORPUS_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/EATD-Corpus/EATD-Corpus'
    PRETRAINED_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models'
    OUTPUT_PATH: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output'
    # Dataset-specific output folders
    OUTPUT_DAIC_WOZ: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/DAIC-WOZ'
    OUTPUT_EXTENDED_DAIC: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/Extended-DAIC'
    OUTPUT_EATD: str = '/content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus'
    TEMP_PATH: str = '/content/temp_extraction'
    NORMALIZER_STATS_PATH: str = None
    IMPUTER_STATS_PATH: str = None
    SAMPLE_RATE: int = 16000
    WINDOW_SEC: float = 10.0
    OVERLAP: float = 0.5
    TARGET_LUFS: float = -23.0
    TARGET_FPS: int = 5
    NUM_FRAMES: int = 16
    FRAME_SIZE: Tuple[int, int] = (224, 224)
    BLUR_THRESHOLD: float = 50.0
    BRIGHTNESS_MIN: int = 80
    BRIGHTNESS_MAX: int = 180
    EMBED_DIM: int = 768
    DEVICE: str = DEVICE

CFG = Config()
os.makedirs(CFG.TEMP_PATH, exist_ok=True)
# Only create parent output if not exists
if not os.path.exists(CFG.OUTPUT_PATH):
    os.makedirs(CFG.OUTPUT_PATH)
print(f'✅ Config loaded. Device: {CFG.DEVICE}')
print(f'📂 Parent output: {CFG.OUTPUT_PATH}')

✅ Config loaded. Device: cuda
📂 Parent output: /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎯 DATASET SELECTOR - Choose which dataset(s) to process        ║
# ║  Set to True/False to enable/disable each dataset               ║
# ╚══════════════════════════════════════════════════════════════════╝

PROCESS_DAIC_WOZ = False       # Original DAIC-WOZ (IDs 300-492)
PROCESS_EXTENDED_DAIC = True   # Extended-DAIC NEW only (IDs 600+)
PROCESS_EATD = False           # EATD-Corpus (IDs t_*)

# Create dataset-specific output folders (only for enabled datasets)
if PROCESS_DAIC_WOZ:
    os.makedirs(CFG.OUTPUT_DAIC_WOZ, exist_ok=True)
if PROCESS_EXTENDED_DAIC:
    os.makedirs(CFG.OUTPUT_EXTENDED_DAIC, exist_ok=True)
if PROCESS_EATD:
    os.makedirs(CFG.OUTPUT_EATD, exist_ok=True)

print('╔══════════════════════════════════════════════════════════════════════════╗')
print('║  🎯 DATASET SELECTION                                                    ║')
print('╠══════════════════════════════════════════════════════════════════════════╣')
if PROCESS_DAIC_WOZ:
    print(f'║  ✅ DAIC-WOZ      → {CFG.OUTPUT_DAIC_WOZ}')
else:
    print('║  ❌ DAIC-WOZ      (disabled)')
if PROCESS_EXTENDED_DAIC:
    print(f'║  ✅ Extended-DAIC → {CFG.OUTPUT_EXTENDED_DAIC}')
else:
    print('║  ❌ Extended-DAIC (disabled)')
if PROCESS_EATD:
    print(f'║  ✅ EATD-Corpus   → {CFG.OUTPUT_EATD}')
else:
    print('║  ❌ EATD-Corpus   (disabled)')
print('╚══════════════════════════════════════════════════════════════════════════╝')


╔══════════════════════════════════════════════════════════════════════════╗
║  🎯 DATASET SELECTION                                                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║  ❌ DAIC-WOZ      (disabled)
║  ❌ Extended-DAIC (disabled)
║  ✅ EATD-Corpus   → /content/drive/MyDrive/DAIC-WOZ_Datasets/H5_OmniFusion_Output/EATD-Corpus
╚══════════════════════════════════════════════════════════════════════════╝


## Cell 6: Import Pipeline

In [10]:
from model_loader import ModelLoader
from pipeline_audio import AudioPreprocessor
from pipeline_text import TextPreprocessor
from pipeline_video_face import VideoPreprocessor, FacePreprocessor
from pipeline_fusion_main import H5OmniFusionPipeline, EXPECTED_SCALAR_FEATURES
print('✅ Pipeline imported')

Utils loaded. Device: cuda
Available: librosa=True, opensmile=True, cv2=True, transformers=True
ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Device: cuda, CUDA: True
Libraries: librosa=True, opensmile=True, praat=True
Libraries: cv2=True, mediapipe=True, transformers=True
Audio Pipeline loaded: 15 classes + AudioPreprocessor
Text Pipeline loaded: 8 classes + TextPreprocessor
Video & Face Pipeline loaded: VideoPreprocessor + FacePreprocessor
✅ Pipeline imported


## Cell 7: Load Models

In [ ]:
loader = ModelLoader(CFG.DEVICE, pretrained_path=CFG.PRETRAINED_PATH)
loader.load_wav2vec2()
loader.load_text_encoder('english')
loader.load_text_encoder('chinese')
loader.load_videomae()
loader.load_face_encoder()
print(f'✅ Models: {list(loader.get_loaded_models().keys())}')

ModelLoader initialized. Pretrained path: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models
Loading Wav2Vec2 from LOCAL: /content/drive/MyDrive/DAIC-WOZ_Datasets/pretrained_models/audio/wav2vec2-large-xlsr-53


## Cell 8: Initialize Preprocessors

In [ ]:
from tqdm.auto import tqdm
audio_proc = AudioPreprocessor(loader)
text_proc = TextPreprocessor(loader, CFG.EMBED_DIM, str(CFG.DEVICE))
video_proc = VideoPreprocessor(loader, CFG.EMBED_DIM)
face_proc = FacePreprocessor(loader, CFG.EMBED_DIM)
print('✅ Preprocessors ready')

## Cell 9: Discover All Participants

In [ ]:
import glob

# DAIC-WOZ (zip files)
daic_zips = sorted(glob.glob(os.path.join(CFG.DAIC_WOZ_PATH, '*_P.zip')))
daic_pids = [os.path.basename(z).replace('.zip', '').replace('_P', '') for z in daic_zips]
daic_pids_set = set(daic_pids)  # For fast overlap checking

# Extended DAIC-WOZ (tar.gz files, NOT folders!)
# FIXED: Look for *.tar.gz files instead of directories
extended_tarballs = sorted(glob.glob(os.path.join(CFG.EXTENDED_DAIC_WOZ_PATH, '*_P.tar.gz')))
# Extract participant IDs: e.g., '600_P.tar.gz' -> '600'
extended_all_pids = [os.path.basename(t).replace('.tar.gz', '').replace('_P', '') for t in extended_tarballs]

# CRITICAL: Exclude IDs that overlap with DAIC-WOZ (300-492 range)
# Only keep NEW Extended-DAIC participants (600+ range)
extended_pids = [pid for pid in extended_all_pids if pid not in daic_pids_set]
extended_files = [t for t in extended_tarballs if os.path.basename(t).replace('.tar.gz', '').replace('_P', '') not in daic_pids_set]

# EATD-Corpus (folders starting with 't')
eatd_folders = sorted([f for f in glob.glob(os.path.join(CFG.EATD_CORPUS_PATH, 't*')) if os.path.isdir(f)])

print(f'╔══════════════════════════════════════════════════════════════════╗')
print(f'║  📊 UNIQUE PARTICIPANT DISCOVERY (No Overlaps!)                  ║')
print(f'╠══════════════════════════════════════════════════════════════════╣')
print(f'║  DAIC-WOZ:        {len(daic_pids):3d} participants (IDs 300-492)           ║')
print(f'║  Extended-DAIC:   {len(extended_pids):3d} NEW participants (IDs 600+)         ║')
print(f'║  EATD-Corpus:     {len(eatd_folders):3d} participants (IDs t_*)              ║')
print(f'╠══════════════════════════════════════════════════════════════════╣')
print(f'║  TOTAL UNIQUE:    {len(daic_pids) + len(extended_pids) + len(eatd_folders):3d} participants                          ║')
print(f'╚══════════════════════════════════════════════════════════════════╝')
print(f'\n(Excluded {len(extended_all_pids) - len(extended_pids)} overlapping IDs from Extended-DAIC)')

## Cell 10: 📂 RESUME FROM CHECKPOINT

**No resume-from-checkpoint needed!** Automatically scans output folder and processes only remaining participants.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  📂 RESUME-FROM-CHECKPOINT MODE                             ║
# ║  Scans EACH dataset's output folder separately.             ║
# ╚══════════════════════════════════════════════════════════════╝

import os

def scan_existing(folder):
    '''Scan folder for existing H5 files'''
    if not os.path.exists(folder):
        return set()
    return set(f.replace('.h5', '') for f in os.listdir(folder) if f.endswith('.h5'))

# Scan each dataset's output folder
existing_daic = scan_existing(CFG.OUTPUT_DAIC_WOZ) if PROCESS_DAIC_WOZ else set()
existing_extended = scan_existing(CFG.OUTPUT_EXTENDED_DAIC) if PROCESS_EXTENDED_DAIC else set()
existing_eatd = scan_existing(CFG.OUTPUT_EATD) if PROCESS_EATD else set()

print(f'📂 Output folders:')
if PROCESS_DAIC_WOZ:
    print(f'   DAIC-WOZ:      {CFG.OUTPUT_DAIC_WOZ} ({len(existing_daic)} processed)')
if PROCESS_EXTENDED_DAIC:
    print(f'   Extended-DAIC: {CFG.OUTPUT_EXTENDED_DAIC} ({len(existing_extended)} processed)')
if PROCESS_EATD:
    print(f'   EATD-Corpus:   {CFG.OUTPUT_EATD} ({len(existing_eatd)} processed)')

# Filter based on DATASET SELECTOR and already-processed
my_daic = [pid for pid in daic_pids if pid not in existing_daic] if PROCESS_DAIC_WOZ else []
my_extended = [pid for pid in extended_pids if pid not in existing_extended] if PROCESS_EXTENDED_DAIC else []
my_eatd = [f for f in eatd_folders if os.path.basename(f) not in existing_eatd] if PROCESS_EATD else []

total_remaining = len(my_daic) + len(my_extended) + len(my_eatd)
total_selected = (len(daic_pids) if PROCESS_DAIC_WOZ else 0) + (len(extended_pids) if PROCESS_EXTENDED_DAIC else 0) + (len(eatd_folders) if PROCESS_EATD else 0)

print(f'\n╔═══════════════════════════════════════════════════════════╗')
print(f'║  🔄 RESUME MODE: {total_remaining}/{total_selected} remaining (selected datasets only)')
if PROCESS_DAIC_WOZ:
    print(f'║  DAIC: {len(my_daic)}/{len(daic_pids)}')
if PROCESS_EXTENDED_DAIC:
    print(f'║  ExtDAIC: {len(my_extended)}/{len(extended_pids)}')
if PROCESS_EATD:
    print(f'║  EATD: {len(my_eatd)}/{len(eatd_folders)}')
print(f'╚═══════════════════════════════════════════════════════════╝')

if total_remaining == 0:
    print('\n✅ All selected participants already processed!')

## Cell 11: 🚀 PROCESS REMAINING PARTICIPANTS

**Resume Mode:** Automatically skips already-processed files.

**Output:** One H5 file per participant (e.g., `355.h5`, `356.h5`, `t_28.h5`)

In [ ]:
from datetime import datetime
import h5py

# ==========================================
# 🛑 PILOT MODE: Set to True to process ONLY 1 participant
PILOT_MODE = False  # Set to False for full processing
# ==========================================

if PILOT_MODE:
    print(f"\n{'!'*60}")
    print("🛑 PILOT MODE ACTIVE: Will process ONLY 1 participant and STOP.")
    print(f"{'!'*60}\n")

pipeline = H5OmniFusionPipeline(audio_proc, text_proc, video_proc, face_proc, CFG)

processed_count = 0
skipped_count = 0
failed_count = 0

print(f'╔═══════════════════════════════════════════════════════════════════╗')
print(f'║  🚀 RESUME-FROM-CHECKPOINT MODE (Dataset-Specific Outputs)       ║')
print(f'╚═══════════════════════════════════════════════════════════════════╝')

def check_pilot_stop(count):
    if PILOT_MODE and count >= 1:
        print(f"\n🛑 PILOT MODE: Limit reached ({count} participant). Stopping.")
        return True
    return False

# ═══════════════════════════════════════════════════════════════════
# PROCESS DAIC-WOZ → CFG.OUTPUT_DAIC_WOZ
# ═══════════════════════════════════════════════════════════════════
if PROCESS_DAIC_WOZ:
    print(f'\n📁 DAIC-WOZ ({len(my_daic)} remaining) → {CFG.OUTPUT_DAIC_WOZ}')
    print('─' * 60)

    for i, pid in enumerate(tqdm(my_daic, desc='DAIC-WOZ')):
        if check_pilot_stop(processed_count): break

        output_file = os.path.join(CFG.OUTPUT_DAIC_WOZ, f'{pid}.h5')

        if os.path.exists(output_file):
            print(f'   ⏭️ {pid}.h5 exists, skipping')
            skipped_count += 1
            continue

        try:
            result = pipeline.process_daic_participant(pid)
            result['dataset'] = 'daic-woz'
            pipeline.save_participant_individual(result, CFG.OUTPUT_DAIC_WOZ)
            processed_count += 1

        except Exception as e:
            print(f'❌ {pid}: {e}')
            failed_count += 1

# ═══════════════════════════════════════════════════════════════════
# PROCESS EXTENDED DAIC-WOZ → CFG.OUTPUT_EXTENDED_DAIC
# ═══════════════════════════════════════════════════════════════════
if PROCESS_EXTENDED_DAIC:
    print(f'\n📁 Extended-DAIC ({len(my_extended)} remaining) → {CFG.OUTPUT_EXTENDED_DAIC}')
    print('─' * 60)

    for i, pid in enumerate(tqdm(my_extended, desc='ExtDAIC')):
        if check_pilot_stop(processed_count): break

        output_file = os.path.join(CFG.OUTPUT_EXTENDED_DAIC, f'{pid}.h5')

        if os.path.exists(output_file):
            print(f'   ⏭️ {pid}.h5 exists, skipping')
            skipped_count += 1
            continue

        try:
            tarball_path = os.path.join(CFG.EXTENDED_DAIC_WOZ_PATH, f'{pid}_P.tar.gz')
            result = pipeline.process_extended_daic_participant(tarball_path)
            result['dataset'] = 'extended-daic'
            pipeline.save_participant_individual(result, CFG.OUTPUT_EXTENDED_DAIC)
            processed_count += 1
        except Exception as e:
            print(f'❌ {pid}: {e}')
            failed_count += 1

# ═══════════════════════════════════════════════════════════════════
# PROCESS EATD-CORPUS → CFG.OUTPUT_EATD
# ═══════════════════════════════════════════════════════════════════
if PROCESS_EATD:
    print(f'\n📁 EATD-Corpus ({len(my_eatd)} remaining) → {CFG.OUTPUT_EATD}')
    print('─' * 60)

    for i, folder in enumerate(tqdm(my_eatd, desc='EATD')):
        if check_pilot_stop(processed_count): break

        pid = os.path.basename(folder)
        output_file = os.path.join(CFG.OUTPUT_EATD, f'{pid}.h5')

        if os.path.exists(output_file):
            print(f'   ⏭️ {pid}.h5 exists, skipping')
            skipped_count += 1
            continue

        try:
            result = pipeline.process_eatd_participant(folder)
            result['dataset'] = 'eatd-corpus'
            pipeline.save_participant_individual(result, CFG.OUTPUT_EATD)
            processed_count += 1

        except Exception as e:
            print(f'❌ {pid}: {e}')
            failed_count += 1

# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════
print(f'\n' + '═' * 60)
print(f'✅ PROCESSING COMPLETE!')
if PILOT_MODE:
    print(f'🛑 PILOT MODE EXECUTION FINISHED')
print(f'═' * 60)
print(f'   ✅ Newly created: {processed_count} files')
print(f'   ⏭️ Skipped (already exist): {skipped_count}')
print(f'   ❌ Failed: {failed_count}')
print(f'\n📂 Output locations:')
if PROCESS_DAIC_WOZ:
    daic_count = len([f for f in os.listdir(CFG.OUTPUT_DAIC_WOZ) if f.endswith('.h5')]) if os.path.exists(CFG.OUTPUT_DAIC_WOZ) else 0
    print(f'   DAIC-WOZ:      {CFG.OUTPUT_DAIC_WOZ} ({daic_count} files)')
if PROCESS_EXTENDED_DAIC:
    ext_count = len([f for f in os.listdir(CFG.OUTPUT_EXTENDED_DAIC) if f.endswith('.h5')]) if os.path.exists(CFG.OUTPUT_EXTENDED_DAIC) else 0
    print(f'   Extended-DAIC: {CFG.OUTPUT_EXTENDED_DAIC} ({ext_count} files)')
if PROCESS_EATD:
    eatd_count = len([f for f in os.listdir(CFG.OUTPUT_EATD) if f.endswith('.h5')]) if os.path.exists(CFG.OUTPUT_EATD) else 0
    print(f'   EATD-Corpus:   {CFG.OUTPUT_EATD} ({eatd_count} files)')


## Cell 12: View Created Files

In [ ]:
import os
import h5py

output_dir = CFG.OUTPUT_PATH
h5_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.h5')])

print(f'📂 {output_dir}')
print(f'   Total files: {len(h5_files)}')
print('\nFirst 10 files:')
for f in h5_files[:10]:
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path) / 1024  # KB
    print(f'   {f}: {size:.1f} KB')

## Cell 13: Inspect One Participant's File

In [ ]:
import h5py
import numpy as np

# Pick first file to inspect
h5_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.h5')])
if h5_files:
    sample_file = os.path.join(output_dir, h5_files[0])
    print(f'Inspecting: {sample_file}')
    print('─' * 50)

    with h5py.File(sample_file, 'r') as f:
        for pid in f.keys():
            grp = f[pid]
            print(f'Participant: {pid}')
            print(f'Dataset: {grp.attrs.get("dataset", "unknown")}')
            print(f'\nFeatures ({len(grp.keys())}):')
            for key in sorted(grp.keys()):
                if isinstance(grp[key], h5py.Dataset):
                    print(f'   {key}: {grp[key].shape}')
                else:
                    print(f'   {key}: [Group]')
else:
    print('No H5 files found')